# Feature engineering for traffic incident hotspot analysis

This notebook prepares the Calgary traffic incident dataset for both spatial
clustering and classification. We parse coordinates, extract temporal features,
standardise spatial dimensions, and create severity/frequency indicators using
functions from `src/data_loader.py`.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler

from src.data_loader import (
    fetch_traffic_incidents, preprocess_dataframe,
    create_clustering_features, create_classification_features,
    CALGARY_LAT, CALGARY_LON,
)

sns.set_style('whitegrid')
%matplotlib inline

## 1. Load and preprocess raw incident data

`fetch_traffic_incidents` loads from the local cache or the Calgary Open Data
API. `preprocess_dataframe` parses `start_dt`, extracts hour/day/month/year,
cleans the quadrant field, converts coordinates to numeric, and filters to
Calgary's bounding box.

In [ ]:
raw_df = fetch_traffic_incidents(limit=100000)
print(f'Raw records: {len(raw_df):,}')
print(f'Columns: {list(raw_df.columns)}')

In [ ]:
df = preprocess_dataframe(raw_df)
print(f'After preprocessing: {len(df):,} rows, {df.shape[1]} columns')
df[['latitude', 'longitude', 'hour', 'day_of_week', 'month', 'year']].describe().round(3)

## 2. Parse and validate coordinates

The preprocessing step already converted lat/lon to numeric and filtered
to Calgary's bounding box (lat 50.8--51.3, lon -114.4 to -113.8). We
verify spatial coverage with a scatter plot.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
ax.scatter(df['longitude'], df['latitude'], s=0.3, alpha=0.2, color='navy')
ax.scatter(CALGARY_LON, CALGARY_LAT, s=100, c='red', marker='x', label='City center')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.set_title('Traffic incident locations in Calgary', fontsize=12)
ax.legend()
plt.tight_layout()
plt.show()

print(f'Latitude range:  [{df["latitude"].min():.4f}, {df["latitude"].max():.4f}]')
print(f'Longitude range: [{df["longitude"].min():.4f}, {df["longitude"].max():.4f}]')

## 3. Temporal feature extraction

`preprocess_dataframe` already extracted `hour`, `day_of_week`, `month`, and
`year`. Let us examine temporal distributions to understand when incidents peak.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

df['hour'].value_counts().sort_index().plot(kind='bar', ax=axes[0], color='steelblue', edgecolor='black')
axes[0].set_title('Incidents by hour of day', fontsize=11)
axes[0].set_xlabel('Hour')
axes[0].set_ylabel('Count')

day_names = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
day_counts = df['day_of_week'].value_counts().sort_index()
axes[1].bar(day_names, day_counts.values, color='teal', edgecolor='black')
axes[1].set_title('Incidents by day of week', fontsize=11)

df['month'].value_counts().sort_index().plot(kind='bar', ax=axes[2], color='coral', edgecolor='black')
axes[2].set_title('Incidents by month', fontsize=11)
axes[2].set_xlabel('Month')

plt.tight_layout()
plt.show()

## 4. Standardise spatial features for clustering

`create_clustering_features` extracts a (n, 2) array of [latitude, longitude].
For KMeans, these need to be scaled so that both axes contribute equally.
DBSCAN with haversine metric operates on radians directly.

In [ ]:
coords = create_clustering_features(df)
print(f'Clustering feature matrix shape: {coords.shape}')

scaler = StandardScaler()
coords_scaled = scaler.fit_transform(coords)

print(f'Scaled mean: {coords_scaled.mean(axis=0).round(6)}')
print(f'Scaled std:  {coords_scaled.std(axis=0).round(6)}')

## 5. Create classification features

`create_classification_features` builds a rich feature set for predicting
whether an hour-quadrant combination has high incident frequency. It adds
cyclical encodings for hour, day, and month; binary flags for weekends
and rush hours; and a `high_incident` binary target.

In [ ]:
X_clf, y_clf = create_classification_features(df)

print(f'Classification feature matrix: {X_clf.shape}')
print(f'Features: {list(X_clf.columns)}')
print(f'\nTarget distribution:')
print(y_clf.value_counts().to_string())
print(f'\nHigh-incident rate: {y_clf.mean():.2%}')

In [ ]:
# Show cyclical encoding works correctly
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sample = X_clf.drop_duplicates(subset=['hour']).sort_values('hour')
axes[0].plot(sample['hour'], sample['hour_sin'], 'o-', label='sin')
axes[0].plot(sample['hour'], sample['hour_cos'], 's-', label='cos')
axes[0].set_xlabel('Hour')
axes[0].set_title('Cyclical hour encoding', fontsize=11)
axes[0].legend()

sample_m = X_clf.drop_duplicates(subset=['month']).sort_values('month')
axes[1].plot(sample_m['month'], sample_m['month_sin'], 'o-', label='sin')
axes[1].plot(sample_m['month'], sample_m['month_cos'], 's-', label='cos')
axes[1].set_xlabel('Month')
axes[1].set_title('Cyclical month encoding', fontsize=11)
axes[1].legend()

plt.tight_layout()
plt.show()

## 6. Create severity indicators

We derive additional severity indicators beyond the binary `high_incident`
target: whether incidents occur during adverse weather months (Nov--Mar)
and an interaction between rush-hour and weekend flags.

In [ ]:
# Winter months indicator (November through March)
X_clf['is_winter'] = X_clf['month'].isin([11, 12, 1, 2, 3]).astype(int)

# Rush-hour on weekday (most congested condition)
X_clf['rush_weekday'] = ((X_clf['is_rush_hour'] == 1) & (X_clf['is_weekend'] == 0)).astype(int)

# Distance from city center (proxy for urban density)
X_clf['dist_from_center'] = np.sqrt(
    (X_clf['latitude'] - CALGARY_LAT) ** 2 +
    (X_clf['longitude'] - CALGARY_LON) ** 2
)

print('New features added: is_winter, rush_weekday, dist_from_center')
X_clf[['is_winter', 'rush_weekday', 'dist_from_center']].describe().round(3)

## 7. Feature correlation heatmap

In [ ]:
corr = X_clf.corr()

fig, ax = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, ax=ax, linewidths=0.3, annot_kws={'size': 7})
ax.set_title('Classification feature correlations', fontsize=13)
plt.tight_layout()
plt.show()

## 8. Save processed datasets

In [ ]:
import os
os.makedirs('../data', exist_ok=True)

df.to_csv('../data/traffic_incidents_preprocessed.csv', index=False)
X_clf['high_incident'] = y_clf.values
X_clf.to_csv('../data/traffic_classification_features.csv', index=False)

print(f'Saved preprocessed data: {len(df):,} rows')
print(f'Saved classification features: {X_clf.shape}')

## Summary

- Parsed and validated geographic coordinates within Calgary's bounding box.
- Extracted hour, day of week, and month from incident timestamps.
- Built cyclical sin/cos encodings so temporal features wrap correctly.
- Created binary flags for weekends, rush hours, and winter months.
- Added a `dist_from_center` feature as a proxy for urban density.
- Standardised spatial features for KMeans clustering.
- The classification target `high_incident` splits incidents into above/below
  median frequency per hour-quadrant.

Next: `03_modeling` runs DBSCAN, KMeans, and classification models.